<a href="https://colab.research.google.com/github/Mrshayan07/llama-3.2-1b-instruct-chatbot-project/blob/main/llama_3_2_1b_instruct_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from huggingface_hub import login

login("YOUR_HUGGING_FACE_API_KEY")

In [35]:
!pip install -q bitsandbytes>=0.46.1

In [36]:
import torch

import gradio as gr

from threading import Thread

from transformers import (

    AutoTokenizer,

    AutoModelForCausalLM,

    BitsAndBytesConfig,

    TextIteratorStreamer

)

In [37]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

In [38]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True

)

In [39]:
tokenizer = AutoTokenizer.from_pretrained(

    model_name

)

In [40]:
model = AutoModelForCausalLM.from_pretrained(

    model_name,

    quantization_config=bnb_config,

    device_map="auto",

    torch_dtype=torch.float16

)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [41]:
from transformers import TextIteratorStreamer
from threading import Thread


# ==========================================
# Chat Function (Streaming Llama Inference)
# ==========================================

def chat(

    history,

    temperature=0.7,

    top_p=0.9,

    do_sample=True,

    max_new_tokens=100

):

    # ======================================
    # Step 1: Convert Chat History to Tokens
    # ======================================

    inputs = tokenizer.apply_chat_template(

        history,

        add_generation_prompt=True,

        return_tensors="pt",

        return_dict=True

    )


    # ======================================
    # Step 2: Move Inputs to GPU
    # ======================================

    inputs = {

        key: value.to(model.device)

        for key, value in inputs.items()

    }


    # ======================================
    # Step 3: Create Streamer
    # ======================================

    streamer = TextIteratorStreamer(

        tokenizer,

        skip_prompt=True,

        skip_special_tokens=True,

        clean_up_tokenization_spaces=False

    )


    # ======================================
    # Step 4: Generation Parameters
    # ======================================

    generation_kwargs = dict(

        **inputs,

        streamer=streamer,

        max_new_tokens=max_new_tokens,

        temperature=temperature,

        top_p=top_p,

        do_sample=do_sample,

        pad_token_id=tokenizer.pad_token_id

    )


    # ======================================
    # Step 5: Background Thread
    # ======================================

    thread = Thread(

        target=model.generate,

        kwargs=generation_kwargs

    )


    # Start Model Generation
    thread.start()



    # ======================================
    # Step 6: Receive Streaming Tokens
    # ======================================

    response = ""


    for text in streamer:

        response += text

        yield response

In [42]:
def respond(
    message,
    history,
    temperature,
    top_p,
    do_sample,
    max_new_tokens
):

    if history is None:
        history = []


    # Add user message
    history.append(
        {
            "role": "user",
            "content": message
        }
    )


    partial_response = ""


    for chunk in chat(
        history,
        temperature,
        top_p,
        do_sample,
        max_new_tokens
    ):

        partial_response = chunk


        temp_history = history.copy()

        temp_history.append(
            {
                "role": "assistant",
                "content": partial_response
            }
        )


        yield "", temp_history


    history.append(
        {
            "role": "assistant",
            "content": partial_response
        }
    )


    yield "", history

In [ ]:
# ==========================================
# Gradio UI - Attractive Version (Gradio 6.0 compatible)
# ==========================================

custom_css = """
#chatbot {
    border-radius: 16px !important;
    box-shadow: 0 4px 20px rgba(0,0,0,0.08);
}

.gradio-container {
    background: linear-gradient(135deg, #f5f7fa 0%, #e8ecf1 100%) !important;
}

#title {
    text-align: center;
    background: linear-gradient(90deg, #6a11cb 0%, #2575fc 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-weight: 800 !important;
    padding: 10px 0;
}

#subtitle {
    text-align: center;
    color: #666;
    margin-bottom: 20px;
}

.send-btn {
    background: linear-gradient(90deg, #6a11cb 0%, #2575fc 100%) !important;
    color: white !important;
    border: none !important;
    font-weight: 600 !important;
}

.clear-btn {
    border: 1px solid #ddd !important;
}

footer {
    display: none !important;
}
"""

theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Poppins"), "sans-serif"]
)

# theme/css yahan se hata diye — ab Blocks() sirf title leta hai
with gr.Blocks(title="Mr. AI Chatbot") as demo:

    gr.Markdown("# 🤖 Mr. AI Chatbot", elem_id="title")
    gr.Markdown("Develop & Design By Shayan Ahmed 👇", elem_id="subtitle")

    chatbot = gr.Chatbot(
    height=500,
    elem_id="chatbot"
)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type the Message here... 💬",
            label="",
            scale=8,
            container=False
        )
        send_btn = gr.Button("Send ➤", scale=1, elem_classes="send-btn")

    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear Chat", elem_classes="clear-btn", size="sm")

    with gr.Accordion("⚙️ Advanced Settings", open=False):
        with gr.Row():
            temperature = gr.Slider(
                minimum=0.1, maximum=2.0, value=0.7, step=0.1,
                label="🌡️ Temperature"
            )
            top_p = gr.Slider(
                minimum=0.1, maximum=1.0, value=0.9, step=0.05,
                label="🎯 Top P"
            )

        with gr.Row():
            do_sample = gr.Checkbox(
                value=True,
                label="🎲 Do Sample"
            )
            max_new_tokens = gr.Slider(
                minimum=10, maximum=1000, value=200, step=10,
                label="📝 Max New Tokens"
            )

    msg.submit(
        respond,
        inputs=[msg, chatbot, temperature, top_p, do_sample, max_new_tokens],
        outputs=[msg, chatbot]
    )

    send_btn.click(
        respond,
        inputs=[msg, chatbot, temperature, top_p, do_sample, max_new_tokens],
        outputs=[msg, chatbot]
    )

    clear_btn.click(lambda: (None, []), outputs=[msg, chatbot])

demo.queue()

# theme aur css ab yahan pass honge, launch() me
demo.launch(
    share=True,
    debug=True,
    theme=theme,
    css=custom_css
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0e407baac0dbf93e06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
